In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [3]:
import json
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from jiwer import cer

from src.data import (
    parse_transcript_file,
    fix_audio_extension,
    ASRDataset,
    build_vocab_from_df,
    make_collate_fn,
)
from src.model_houlsby import HuBERT_CTC_Houlsby
from src.train_utils import (
    set_seed,
    train_epoch,
    evaluate,
    save_checkpoint,
    count_parameters,
)
from src.eval_utils import (
    run_inference,
    save_results_json,
    save_predictions_csv,
)

In [4]:
CONFIG = {
    "model_name": "utter-project/mHuBERT-147-base-3rd-iter",
    "data_size": "10min",
    "lang": "swa",
    "batch_size": 8,
    "grad_accumulation": 4,
    "learning_rate": 1e-4,
    "weight_decay": 1e-6,
    "target_iterations": 1500,
    "eval_every": 5,
    "save_every": 100,
    "experiment_name": "mhubert_houlsby_swa_10min_b32",
    "adapter_bottleneck": 32,
    "adapter_dropout": 0.1,
}

In [5]:
DATA_DIR = PROJECT_ROOT / "data" / "commonvoice" / "swa"
RESULTS_DIR = PROJECT_ROOT / "results" / "houlsby"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints" / "houlsby"
FIGURES_DIR = PROJECT_ROOT / "figures" / "houlsby"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [7]:
df_train_10min = parse_transcript_file(DATA_DIR / "transcript_10min_train.txt")
df_train_1h = parse_transcript_file(DATA_DIR / "transcript_1h_train.txt")
df_dev = parse_transcript_file(DATA_DIR / "transcript_10min_dev.txt")
df_test = parse_transcript_file(DATA_DIR / "transcript_10min_test.txt")

df_train_10min = fix_audio_extension(df_train_10min)
df_train_1h = fix_audio_extension(df_train_1h)
df_dev = fix_audio_extension(df_dev)
df_test = fix_audio_extension(df_test)

Parsed 92 samples from /users/eleves-a/2022/stephane.eilles-chan-way/Desktop/4A/NLP/ml-superb/data/commonvoice/swa/transcript_10min_train.txt
Parsed 589 samples from /users/eleves-a/2022/stephane.eilles-chan-way/Desktop/4A/NLP/ml-superb/data/commonvoice/swa/transcript_1h_train.txt
Parsed 103 samples from /users/eleves-a/2022/stephane.eilles-chan-way/Desktop/4A/NLP/ml-superb/data/commonvoice/swa/transcript_10min_dev.txt
Parsed 100 samples from /users/eleves-a/2022/stephane.eilles-chan-way/Desktop/4A/NLP/ml-superb/data/commonvoice/swa/transcript_10min_test.txt


In [8]:
if CONFIG["data_size"] == "10min":
    df_train = df_train_10min
else:
    df_train = df_train_1h

vocab = build_vocab_from_df(df_train)
idx_to_char = {v: k for k, v in vocab.items()}

In [9]:
if CONFIG["data_size"] == "10min":
    df_train = df_train_10min
else:
    df_train = df_train_1h

vocab = build_vocab_from_df(df_train)
idx_to_char = {v: k for k, v in vocab.items()}

In [10]:
AUDIO_DIR = DATA_DIR / "wav"

train_dataset = ASRDataset(df_train, AUDIO_DIR)
dev_dataset = ASRDataset(df_dev, AUDIO_DIR)
test_dataset = ASRDataset(df_test, AUDIO_DIR)

collate_fn = make_collate_fn(vocab)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)

Loaded 92 samples from            utt_id     audio_filename  \
0   cv_swa_000000  cv_swa_000000.wav   
1   cv_swa_000001  cv_swa_000001.wav   
2   cv_swa_000002  cv_swa_000002.wav   
3   cv_swa_000003  cv_swa_000003.wav   
4   cv_swa_000004  cv_swa_000004.wav   
..            ...                ...   
87  cv_swa_000087  cv_swa_000087.wav   
88  cv_swa_000088  cv_swa_000088.wav   
89  cv_swa_000089  cv_swa_000089.wav   
90  cv_swa_000090  cv_swa_000090.wav   
91  cv_swa_000091  cv_swa_000091.wav   

                                                 text  
0   wachambuzi wa soka wanamtaja messi kama nyota ...  
1   romario aliingia kwenye orodha ya wachezaji wa...  
2       sote twesangaa twelipomuona mwalimu ali apika  
3                  inajulikana kama shina la warangi.  
4   mwidhi aliyejepa simu amekatiwa kifungo cha mw...  
..                                                ...  
87  mwana ndhuri ni ule apendae kuridhiya vadhadhi...  
88  "mke wangu ambadilika sana, tabia zimebadili

In [11]:
num_train_samples = len(train_dataset)
effective_batch = CONFIG["batch_size"] * CONFIG["grad_accumulation"]
iterations_per_epoch = num_train_samples / effective_batch
num_epochs = int((CONFIG["target_iterations"] / iterations_per_epoch) + 0.9999)

CONFIG["num_epochs"] = num_epochs
print("num_epochs =", num_epochs)

num_epochs = 522


In [12]:
model = HuBERT_CTC_Houlsby(
    vocab_size=len(vocab),
    model_name=CONFIG["model_name"],
    adapter_bottleneck=CONFIG["adapter_bottleneck"],
    adapter_dropout=CONFIG["adapter_dropout"],
).to(device)

count_parameters(model)

Total params: 104.7M
Trainable params: 10.29M (9.8%)
Frozen params: 94.4M


(104659634, 10287922)

In [13]:
criterion = nn.CTCLoss(blank=vocab["<blank>"], zero_infinity=True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=10,
)

In [14]:
best_dev_loss = float("inf")
history = {
    "train_loss": [],
    "dev_loss": [],
    "lr": [],
    "epoch": [],
}

start_time = time.time()
total_iterations = 0
best_ckpt_path = CHECKPOINTS_DIR / f"{CONFIG['experiment_name']}_best.pt"

for epoch in range(CONFIG["num_epochs"]):
    train_loss = train_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
        CONFIG["grad_accumulation"],
    )

    total_iterations += len(train_loader) // CONFIG["grad_accumulation"]

    if (epoch + 1) % CONFIG["eval_every"] == 0 or epoch == 0 or epoch == CONFIG["num_epochs"] - 1:
        dev_loss = evaluate(model, dev_loader, criterion, device)
        scheduler.step(dev_loss)
    else:
        dev_loss = history["dev_loss"][-1] if history["dev_loss"] else float("inf")

    history["train_loss"].append(train_loss)
    history["dev_loss"].append(dev_loss)
    history["lr"].append(optimizer.param_groups[0]["lr"])
    history["epoch"].append(epoch + 1)

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            path=best_ckpt_path,
            dev_loss=dev_loss,
            config=CONFIG,
            vocab=vocab,
        )

Training:   0%|          | 0/12 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

Training:   0%|          | 0/12 [00:00<?, ?it/s]

Training:   0%|          | 0/12 [00:00<?, ?it/s]

Training:   0%|          | 0/12 [00:00<?, ?it/s]

...

Training:   0%|          | 0/12 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

Training:   0%|          | 0/12 [00:00<?, ?it/s]

Training:   0%|          | 0/12 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

In [15]:
history_path = RESULTS_DIR / f"{CONFIG['experiment_name']}_history.json"
with open(history_path, "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2, ensure_ascii=False)

In [16]:
checkpoint = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

predictions, references = run_inference(
    model=model,
    loader=test_loader,
    device=device,
    idx_to_char=idx_to_char,
    vocab=vocab,
)

char_error_rate = cer(references, predictions)
elapsed_total = (time.time() - start_time) / 60

/tmp/ipykernel_2095552/2564674195.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(best_ckpt_path, map_location=device)


Test inference:   0%|          | 0/13 [00:00<?, ?it/s]

In [17]:
results = {
    "experiment_name": CONFIG["experiment_name"],
    "model_name": CONFIG["model_name"],
    "lang": CONFIG["lang"],
    "data_size": CONFIG["data_size"],
    "num_train_samples": len(train_dataset),
    "num_epochs": CONFIG["num_epochs"],
    "target_iterations": CONFIG["target_iterations"],
    "actual_iterations": total_iterations,
    "best_dev_loss": best_dev_loss,
    "cer": char_error_rate,
    "training_time_minutes": elapsed_total,
}

save_results_json(
    results,
    RESULTS_DIR / f"{CONFIG['experiment_name']}_results.json",
)

save_predictions_csv(
    references,
    predictions,
    RESULTS_DIR / f"{CONFIG['experiment_name']}_predictions.csv",
)

print(results)

{'experiment_name': 'mhubert_houlsby_swa_10min_b32', 'model_name': 'utter-project/mHuBERT-147-base-3rd-iter', 'lang': 'swa', 'data_size': '10min', 'num_train_samples': 92, 'num_epochs': 522, 'target_iterations': 1500, 'actual_iterations': 1566, 'best_dev_loss': 1.0224384848888104, 'cer': 0.22243602078121993, 'training_time_minutes': 27.19651883840561}
